# A6: Naive RAG vs Contextual Retrieval with LangChain

Chapter 8 is loaded from the PDF in this folder. This notebook also recreates the cleaned text file and QA JSON inside the notebook itself.

In [ ]:
# %pip install -q langchain langchain-community langchain-groq langchain-text-splitters sentence-transformers faiss-cpu rouge-score python-dotenv streamlit pypdf

In [ ]:
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List

import pandas as pd
from dotenv import load_dotenv
from rouge_score import rouge_scorer

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(Path.cwd() / ".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing in .env")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
ANSWER_DIR = BASE_DIR / "answer"
APP_DIR = BASE_DIR / "app"
PDF_PATH = BASE_DIR / "Chapter-8 RAG assignment.pdf"
TEXT_PATH = DATA_DIR / "chapter8_cleaned.txt"
QA_PATH = DATA_DIR / "qa_pairs.json"
CONTEXT_CACHE_PATH = DATA_DIR / "chapter8_contextual_chunks.json"
OUTPUT_JSON_PATH = ANSWER_DIR / "response-st-126018-chapter-8.json"

DATA_DIR.mkdir(exist_ok=True)
ANSWER_DIR.mkdir(exist_ok=True)
APP_DIR.mkdir(exist_ok=True)

GENERATION_MODEL_NAME = "llama-3.1-8b-instant"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [ ]:
pdf_path = str(PDF_PATH)
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Number of pages:", len(pages))
print("\nFirst 1000 characters from page 1:\n")
print(pages[0].page_content[:1000])

full_text = "\n\n".join(page.page_content for page in pages)

def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\t", " ")
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

cleaned_text = clean_text(full_text)
print("Cleaned text length:", len(cleaned_text))
print(cleaned_text[:1500])

with open(TEXT_PATH, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("Saved cleaned text to", TEXT_PATH)
chapter_text = cleaned_text

In [ ]:
qa_pairs = json.loads(r'''[
  {
    "question": "What is the primary purpose of the self-attention mechanism in a transformer?",
    "ground_truth_answer": "Self-attention allows a model to build contextual representations of a token by integrating information from other tokens in the sequence.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "In the attention mechanism, what roles do the query, key, and value vectors play?",
    "ground_truth_answer": "The query represents the current token being compared, the key represents tokens used for similarity comparison, and the value contains the information that is weighted and combined in the output.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "Why is a scaling factor used in the dot product of query and key vectors?",
    "ground_truth_answer": "The dot product is scaled by the square root of the key dimension to prevent large values that could cause unstable gradients during training.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "Why do transformers use multi-head attention instead of a single attention head?",
    "ground_truth_answer": "Multi-head attention allows the model to attend to different types of relationships in the sequence simultaneously using multiple attention heads.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What components are included in a standard transformer block?",
    "ground_truth_answer": "A transformer block includes a multi-head self-attention layer, a feedforward network, residual connections, and layer normalization.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is the residual stream in a transformer block?",
    "ground_truth_answer": "The residual stream is the pathway where token representations are passed through layers while each component reads from and adds its output back to the stream.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "How are positional embeddings combined with token embeddings in a transformer?",
    "ground_truth_answer": "Positional embeddings are added to token embeddings so the model can represent both the token identity and its position in the sequence.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is the architecture of the feedforward layer in a transformer block?",
    "ground_truth_answer": "The feedforward layer is a two-layer fully connected network applied independently to each token representation.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is the purpose of layer normalization in transformers?",
    "ground_truth_answer": "Layer normalization stabilizes training by normalizing activations so they have a consistent scale across the network.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "Why is masking used in the self-attention of causal language models?",
    "ground_truth_answer": "Masking prevents tokens from attending to future tokens so the model only uses previous context when predicting the next token.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What components make up the language modeling head?",
    "ground_truth_answer": "The language modeling head consists of a linear projection called the unembedding layer followed by a softmax to produce probabilities over the vocabulary.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is weight tying in transformer language models?",
    "ground_truth_answer": "Weight tying refers to sharing the same weight matrix between the token embedding layer and the final unembedding layer.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "How does top-k sampling work during text generation?",
    "ground_truth_answer": "Top-k sampling restricts the probability distribution to the k most likely tokens and randomly samples the next token from that subset.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is the intuition behind top-p or nucleus sampling?",
    "ground_truth_answer": "Top-p sampling selects the smallest set of tokens whose cumulative probability exceeds a threshold p and samples from that set.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What training objective is commonly used for large language models?",
    "ground_truth_answer": "Large language models are typically trained using cross-entropy loss to maximize the probability of the correct next token.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "How does the KV cache improve inference efficiency?",
    "ground_truth_answer": "The KV cache stores key and value vectors from previous tokens so they do not need to be recomputed during autoregressive generation.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "Why is attention sometimes called a token-mixing component?",
    "ground_truth_answer": "Attention is called token-mixing because it integrates information from other tokens into the representation of the current token.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is a decoder-only transformer model?",
    "ground_truth_answer": "A decoder-only transformer is a unidirectional model that predicts tokens autoregressively using only the decoder architecture.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is a limitation of absolute positional embeddings?",
    "ground_truth_answer": "Absolute positional embeddings may generalize poorly to positions near the maximum sequence length because those positions appear less frequently in training.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  },
  {
    "question": "What is the purpose of the logit lens tool?",
    "ground_truth_answer": "The logit lens is an interpretability method that applies the final unembedding layer to intermediate activations to analyze what the model is predicting at different layers.",
    "naive_rag_answer": "",
    "contextual_retrieval_answer": ""
  }
]''')

with open(output_dir / "qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved QA pairs to", output_dir / "qa_pairs.json")
print("Number of QA pairs:", len(qa_pairs))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=180, separators=["\n\n", "\n", ". ", " ", ""])
naive_chunks = text_splitter.split_text(chapter_text)
naive_documents = [Document(page_content=chunk, metadata={"source": PDF_PATH.name, "chunk_id": i, "retrieval_type": "naive"}) for i, chunk in enumerate(naive_chunks)]

embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, model_kwargs={"device": "cpu"}, encode_kwargs={"normalize_embeddings": True})
naive_vectorstore = FAISS.from_documents(naive_documents, embedding_model)
naive_retriever = naive_vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatGroq(api_key=GROQ_API_KEY, model=GENERATION_MODEL_NAME, temperature=0)
context_prompt = ChatPromptTemplate.from_template("""You are enriching a chunk for retrieval.\n\nTitle: {title}\nDocument excerpt:\n{document_excerpt}\n\nChunk:\n{chunk}\n\nWrite 1-2 concise sentences explaining how this chunk relates to the full document. Start with: 'This chunk from {title} discusses ...'""")
context_chain = context_prompt | llm | StrOutputParser()

def enrich_chunk(chunk: str, document: str, title: str) -> str:
    context = context_chain.invoke({"title": title, "document_excerpt": document[:4000], "chunk": chunk}).strip()
    return f"{context}\n\n{chunk}"

if CONTEXT_CACHE_PATH.exists():
    cached_items = json.loads(CONTEXT_CACHE_PATH.read_text(encoding="utf-8"))
    contextual_documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in cached_items]
else:
    contextual_documents = []
    for doc in naive_documents:
        contextual_documents.append(Document(page_content=enrich_chunk(doc.page_content, chapter_text, "Chapter 8: Transformers"), metadata={**doc.metadata, "retrieval_type": "contextual"}))
        time.sleep(0.2)
    CONTEXT_CACHE_PATH.write_text(json.dumps([{"page_content": d.page_content, "metadata": d.metadata} for d in contextual_documents], indent=2), encoding="utf-8")

contextual_vectorstore = FAISS.from_documents(contextual_documents, embedding_model)
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 4})

qa_prompt = ChatPromptTemplate.from_template("""You are a careful question answering assistant for Chapter 8 about transformers. Answer using only the retrieved context.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:""")
qa_chain = qa_prompt | llm | StrOutputParser()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(f"[Source: {doc.metadata['source']} | chunk_id={doc.metadata['chunk_id']}]\n{doc.page_content}" for doc in docs)

def answer_with_rag(question: str, retriever) -> Dict:
    docs = retriever.invoke(question)
    context = format_docs(docs)
    answer = qa_chain.invoke({"context": context, "question": question}).strip()
    return {"answer": answer, "source_documents": docs}

results = []
for item in qa_pairs:
    naive_result = answer_with_rag(item["question"], naive_retriever)
    contextual_result = answer_with_rag(item["question"], contextual_retriever)
    results.append({"question": item["question"], "ground_truth_answer": item["ground_truth_answer"], "naive_rag_answer": naive_result["answer"], "contextual_retrieval_answer": contextual_result["answer"]})

OUTPUT_JSON_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
def avg_rouge(rows, key):
    totals = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for row in rows:
        scores = scorer.score(row["ground_truth_answer"], row[key])
        for metric in totals:
            totals[metric] += scores[metric].fmeasure
    return {metric: round(value / len(rows), 4) for metric, value in totals.items()}

naive_scores = avg_rouge(results, "naive_rag_answer")
contextual_scores = avg_rouge(results, "contextual_retrieval_answer")
pd.DataFrame([
    {"Method": "Naive RAG", "ROUGE-1": naive_scores["rouge1"], "ROUGE-2": naive_scores["rouge2"], "ROUGE-L": naive_scores["rougeL"]},
    {"Method": "Contextual Retrieval", "ROUGE-1": contextual_scores["rouge1"], "ROUGE-2": contextual_scores["rouge2"], "ROUGE-L": contextual_scores["rougeL"]},
])

In [ ]:
app_code = r'''import json, os
from pathlib import Path
import streamlit as st
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
BASE_DIR = Path(__file__).resolve().parents[1]
DATA_DIR = BASE_DIR / "data"
load_dotenv(BASE_DIR / ".env")
api_key = os.getenv("GROQ_API_KEY")
cached_items = json.loads((DATA_DIR / "chapter8_contextual_chunks.json").read_text(encoding="utf-8"))
documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in cached_items]
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cpu"}, encode_kwargs={"normalize_embeddings": True})
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatGroq(api_key=api_key, model="llama-3.1-8b-instant", temperature=0)
prompt = ChatPromptTemplate.from_template("Context:\n{context}\n\nQuestion: {question}\nAnswer:")
chain = prompt | llm | StrOutputParser()
def format_docs(docs):
    return "\n\n".join(f"[Source: {doc.metadata['source']} | chunk_id={doc.metadata['chunk_id']}]\n{doc.page_content}" for doc in docs)
st.title("Chapter 8 QA Chatbot")
question = st.text_input("Ask a question about transformers:")
if question:
    docs = retriever.invoke(question)
    answer = chain.invoke({"context": format_docs(docs), "question": question})
    st.write(answer)
    for doc in docs:
        st.markdown(f"**chunk_id={doc.metadata['chunk_id']}** from `{doc.metadata['source']}`")
        st.code(doc.page_content[:1200])
'''
(APP_DIR / "app.py").write_text(app_code, encoding="utf-8")
(APP_DIR / "requirements.txt").write_text("streamlit\nlangchain\nlangchain-community\nlangchain-groq\nsentence-transformers\nfaiss-cpu\npython-dotenv", encoding="utf-8")
print("Wrote app/app.py and app/requirements.txt")